# <center> Deep Learning Coursework </center>
## <center> Audio Text Transformer Implementation </center>

Explaination of Audio Text Transformer

Imports required

In [ ]:
import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision import transforms

In [ ]:
%run audio-text_transformer.py --learning-rate 1e-4 --seed 42

In [ ]:
%run audio-text_transformer_pos.py --learning-rate 1e-4 --seed 42 --model-name baseline

In [ ]:
%run audio-text_transformer_pos.py --learning-rate 1e-4 --seed 42 --pos-embed --model-name pos_embed

In [ ]:
def evaluate(weights_path, use_pos, split="val"):
    audio, text, labels = load_features(Path("/data/COMS30017"), split)
    ds = AudioTextDataset(audio, text, labels, training=False)
    loader = DataLoader(ds, batch_size=128, shuffle=False)
    a0, t0, _ = ds[0]

    m = AudioTextTransformer(audio_dim=a0.shape[-1], text_dim=t0.shape[-1],
                             d_model=MODEL_DIM, num_heads=4,
                             mlp_dim=TRANSFORMER_MLP_DIM, dropout=DROPOUT,
                             use_pos=use_pos)
    m.load_state_dict(torch.load(weights_path, map_location=DEVICE, weights_only=True))
    m.to(DEVICE).eval()

    correct = total = 0
    with torch.no_grad():
        for a, t, y in loader:
            preds = m(a.to(DEVICE), t.to(DEVICE)).argmax(-1).cpu()
            correct += (preds == y).sum().item()
            total += len(y)
    return correct / total

In [ ]:
import shutil
results = {"baseline": [], "pos": []}
args_common = "--learning-rate 1e-4"

for seed in [42, 43, 44]:
    get_ipython().run_line_magic("run",
        f"audio-text_transformer_pos.py {args_common} --seed {seed} --model-name baseline_s{seed}")
    shutil.copy("transformer_weights.pth", f"baseline_s{seed}.pth")
    results["baseline"].append(evaluate(f"baseline_s{seed}.pth", use_pos=False))

    get_ipython().run_line_magic("run",
        f"audio-text_transformer_pos.py {args_common} --seed {seed} --pos-embed --model-name pos_s{seed}")
    shutil.copy("transformer_pos_weights.pth", f"pos_s{seed}.pth")
    results["pos"].append(evaluate(f"pos_s{seed}.pth", use_pos=True))

for name, accs in results.items():
    print(f"{name}: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}  (runs: {[round(a*100,2) for a in accs]})")

In [ ]:
weights = torch.load(
    "transformer_weights.pth",
    map_location="cpu",
    weights_only=True
)

for name, tensor in weights.items():
    print(f"Layer: {name}")
    print(f"Shape: {tensor.shape}")
    print(f"Weights:\n{tensor}")
    print("-" * 50)

pos

In [ ]:
weights = torch.load(
    "transformer_pos_weights.pth",
    map_location="cpu",
    weights_only=True
)

for name, tensor in weights.items():
    print(f"Layer: {name}")
    print(f"Shape: {tensor.shape}")
    print(f"Weights:\n{tensor}")
    print("-" * 50)

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_root = Path("/data/COMS30017")

# Load validation features
val_audio, val_text, val_labels = load_features(
    dataset_root, "val"
)

# Create validation dataset
val_dataset = AudioTextDataset(
    val_audio, val_text, val_labels, training=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

# Get feature dimensions from the dataset
first_audio, first_text, _ = val_dataset[0]

# Initialise your actual model
model = AudioTextTransformer(
    audio_dim=first_audio.shape[-1],
    text_dim=first_text.shape[-1],
    d_model=MODEL_DIM,
    num_heads=4,
    mlp_dim=TRANSFORMER_MLP_DIM,
    dropout=DROPOUT
)

# Load trained model weights
model.load_state_dict(
    torch.load(
        "transformer_weights.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

model = model.to(DEVICE)

# Evaluation setup
criterion = nn.CrossEntropyLoss()
model.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, text, labels in val_loader:

        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        # Get model predictions
        logits = model(audio, text)

        # Calculate loss
        loss = criterion(logits, labels)
        total_loss += loss.item()

        # Get predicted classes
        predictions = logits.argmax(dim=-1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = np.mean(
    np.array(all_preds) == np.array(all_labels)
)

average_loss = total_loss / len(val_loader)

print(f"Validation Loss: {average_loss:.5f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Total Samples: {len(all_labels)}")

pos

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_root = Path("/data/COMS30017")

# Load validation features
val_audio, val_text, val_labels = load_features(
    dataset_root, "val"
)

# Create validation dataset
val_dataset = AudioTextDataset(
    val_audio, val_text, val_labels, training=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

# Get feature dimensions from the dataset
first_audio, first_text, _ = val_dataset[0]

# Initialise your actual model
model = AudioTextTransformer(
    audio_dim=first_audio.shape[-1],
    text_dim=first_text.shape[-1],
    d_model=MODEL_DIM,
    num_heads=4,
    mlp_dim=TRANSFORMER_MLP_DIM,
    dropout=DROPOUT,
    use_pos=True
)

# Load trained model weights
model.load_state_dict(
    torch.load(
        "transformer_pos_weights.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

model = model.to(DEVICE)

# Evaluation setup
criterion = nn.CrossEntropyLoss()
model.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, text, labels in val_loader:

        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        # Get model predictions
        logits = model(audio, text)

        # Calculate loss
        loss = criterion(logits, labels)
        total_loss += loss.item()

        # Get predicted classes
        predictions = logits.argmax(dim=-1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = np.mean(
    np.array(all_preds) == np.array(all_labels)
)

average_loss = total_loss / len(val_loader)

print(f"Validation Loss: {average_loss:.5f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Total Samples: {len(all_labels)}")

In [ ]:
print(evaluate("pos_s42.pth", use_pos=True, split="test"))
print(evaluate("baseline_s42.pth", use_pos=False, split="test"))